# Scraping Weather Data Using Meteostat
- Using nearby station and daily weather endpoints to find the closest weather stations to our 6 airports, and grab daily weather data from 12/1/24 - 2/28/25

## Documentation

Meteostat: https://dev.meteostat.net/api

Nearby stations endpoint: https://dev.meteostat.net/api/stations/nearby

Weather daily data endpoint: https://dev.meteostat.net/api/stations/daily

Airport latitude and longitudes: https://www.latlong.net/

In [1]:
import pandas as pd
import requests
import requests_cache
import time

# 1. Defining API Key

In [2]:
def read_key(keyfile):
    with open(keyfile) as f:
        return f.readline().strip("\n")

key = read_key("Meteostat API Key.txt") # this file (in gitignore) holds my personal API key

# 2. Initial Request to Daily Weather - Understand Structure of Call

In [ ]:
# getting daily weather data from station 10637 from 12/1/2024 - 3/1/2025

session = requests_cache.CachedSession('weather_cache') # caching to prevent duplicate requests to server (only get 500 requests per month with free account)

url = "https://meteostat.p.rapidapi.com/stations/daily" # endpoint

querystring = {"station":"10637","start":"2024-12-01","end":"2025-03-01"} # defining parameters, station = station ID found in the file provided on GitHub, quering over specific time frame

headers = {
	"x-rapidapi-key": key,
	"x-rapidapi-host": "meteostat.p.rapidapi.com"
}

response = session.get(url, headers=headers, params=querystring) # making API call

result = response.json() # converting response to json, showing the structure of the result

# 3. Initial Request to Nearby Stations - Understanding Structure

In [ ]:
url = "https://meteostat.p.rapidapi.com/stations/nearby"

querystring = {"lat":"51.5085","lon":"-0.1257"}

headers = {
	"x-rapidapi-key": key,
	"x-rapidapi-host": "meteostat.p.rapidapi.com"
}

response = session.get(url, headers=headers, params=querystring)

# 4. Defining Function to Calculate Minimum Distance
- Input the latitude and longitude of a location to find the closest weather station
- Returns a tuple with the station id of the closest station, and the distance in km

In [ ]:
# defining a function that finds the closest weather station to the inputted latitude and longitude

def find_closest_station(air_lat, air_long):

    station_url = "https://meteostat.p.rapidapi.com/stations/nearby"

    querystring = {"lat":air_lat,"lon":air_long}

    headers = {
	    "x-rapidapi-key": key,
	    "x-rapidapi-host": "meteostat.p.rapidapi.com"
    }

    response = session.get(station_url, headers=headers, params=querystring)
    result = response.json() # the result is a dictionary, and within the outer dictionary is the key data
    # the key data holds all of the information about the station closest (smallest distance by latitude and longitude)

    data = result.get("data") # data is a list, where each element of the list is a dictionary of the closest matching stations
    # the first dictionary of the list is the station that is closest to the input lat/long

    # initializing these variables so if no match is made, these become None rather than the code breaking
    station_id = None
    station_name = None
    
    if data: # if the list isn't empty

        best_match = data[0] # best_match is the first dictionary, the station that's the best match
        
        if best_match is not None:
        
            station_id = best_match.get("id") # within this best_match dictionary is a key named id, which holds the station id

            # also want to grab the name of the station to ensure the match makes sense
            name_dict = best_match.get("name") # a key within best_match called name, which holds naming information about the station
            # the value of name is another dictionary

            if name_dict is not None:
                
                station_name = name_dict.get("en") # the key en holds the name of the station
    
    time.sleep(10) # waiting between calls
    
    return station_id, station_name

In [17]:
# testing function on ORD coordinates

air_lat = 41.978611
air_long = -87.904724

station_id, station_name = find_closest_station(air_lat, air_long)

# 6. Finding Station IDs for All Airports

Making comparisons across 6 airports:
- EWR: Newark Liberty International Airport
- BOS: Boston Logan International Airport
- LGA: LaGuardia Airport
- SFO: San Francisco International Airport
- DFW: Dallas/Fort Worth International Airport
- ORD: O’Hare International Airport

These airports were selected based on a list of airports with the most cancellations.

In [11]:
# creating a dictionary of each airport we want weather data on and location info as a tuple (latitude, longitude)

airport_dict = {
    "EWR": (40.689491, -74.174538), 
    "BOS": (42.365589, -71.010025), 
    "LGA": (40.776863, -73.874069),
    "SFO": (37.615223, -122.389977),
    "DFW": (32.897480, -97.040443),
    "ORD": (41.978611, -87.904724)
}

In [ ]:
station_dict = {} # initializing a dictionary to hold the airport code and the station id that is matched (based on the minimum distance)
name_list = [] # initializing a list to hold all of the station names to the match makes sense

for key, value in airport_dict.items(): # looping through the airports we want to collect weather data on
    air_lat, air_long = value # unpacking the tuple of latitude and longitude
    station_id, station_name = find_closest_station(air_lat, air_long) # calling the find_closest_station function to get the station id match
    station_dict[key] = station_id # populating the dictionary with airport code and matched station
    name_list.append(station_name) # adding station name to list

All of the matches look accurate, ready to call the API to grab weather data for each location

# 8. Creating Dictionary to Hold All Weather Data for All Locations

In [ ]:
url = "https://meteostat.p.rapidapi.com/stations/daily"

weather_dict = {}

for airport, station_id in station_dict.items():

    querystring = {"station":station_id,"start":"2024-12-01","end":"2025-02-28"}

    # need to hid my api key
    headers = {
        "x-rapidapi-key": key,
        "x-rapidapi-host": "meteostat.p.rapidapi.com"
    }

    response = session.get(url, headers=headers, params=querystring)

    if response.status_code == 200:
    
        weather_result = response.json()["data"] # weather data within an outside dictionary, only want to store this inner dictionary
        weather_dict[airport] = weather_result

    time.sleep(10)

# 9. Data Wrangling of weather_dict
- Want a dataframe for each location, where each of the days weather data is collected is a row
- Columns with date, and all weather-related information

In [20]:
list_df = [] # initializing a list to hold each individual df

for airport, info in weather_dict.items(): # looping through the weather dictionary to convert each airport's dictionary to a dataframe
    df = pd.DataFrame(info) # converting dictinary[info] to dataframe - info holds all the weather info, don't need the other stuff in the dictionary
    df["airport"] = airport # adding a column for the airport name
    list_df.append(df) # adding dictionary to list

In [ ]:
weather_df = pd.concat(list_df, ignore_index=True, sort=False) # concatenating all of the dataframes created by the loop into one big dataframe
weather_df = weather_df.convert_dtypes() # using this to ensure any missing values are coded properly as NA
weather_df.to_csv("Airport Weather Data.csv", index=False) # saving df as csv